In [1]:
import ast
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties, fontManager
from matplotlib.patches import Rectangle

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

label = 'pident_90'
os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score
from tqdm import tqdm
import multiprocessing as mp
import os

plasmidness_col = "average plasmid fraction-pident_90"
N_BOOT = 1000
RANDOM_SEED = 42
N_WORKERS = mp.cpu_count() // 2
BASE_FOLDER = '/active-data/analysis_results/chr_pla/genus/recluster_result'
OUTPUT_SUMMARY_CSV = os.path.join(BASE_FOLDER, "genus_bootstrap_summary.csv")
CATEGORY_LABEL = "pident_90"
RAW_PLASMIDNESS_THRESHOLD = 0.3

def process_replicon_data(fraction_data, label=CATEGORY_LABEL):
    typ_chr = fraction_data[fraction_data[f'category-{label}'] == 'typical chromosome']
    min_size = min(typ_chr['size'])
    tra_rep = fraction_data[fraction_data[f'category-{label}'] == 'intermediate replicon']
    if tra_rep.empty:
        return min_size * 0.5
    else:
        tra_max = max(tra_rep['size'])
        return (min_size + tra_max)/2

def run_hier_auto_cluster(X_in):
    Z = linkage(X_in, method='average')
    merge_dist = Z[-100:, 2]
    diff1 = np.diff(merge_dist)
    diff2 = np.diff(diff1)
    idx_max_diff2 = np.argmax(np.abs(diff2))
    z_a = merge_dist[idx_max_diff2]
    z_b = merge_dist[idx_max_diff2 + 1]
    z_c = merge_dist[idx_max_diff2 + 2]
    current_t = (z_a + z_b + z_c) / 3
    labels = fcluster(Z, t=current_t, criterion='distance')
    n_cls = len(np.unique(labels))
    sil = silhouette_score(X_in, labels) if n_cls >= 2 else np.nan
    return current_t, n_cls, sil, labels

def _boot_worker(X, seed, result_queue):
    rng = np.random.default_rng(seed)
    n_sample = X.shape[0]
    idx_boot = rng.choice(n_sample, size=n_sample, replace=True)
    X_boot = X[idx_boot, :]
    _, nc, _, _ = run_hier_auto_cluster(X_boot)
    result_queue.put(nc)

def bootstrap_test(genus_name, X, n_boot, seed, n_workers=N_WORKERS):
    rng_main = np.random.default_rng(seed)
    real_t, real_ncls, real_sil, real_labels = run_hier_auto_cluster(X)

    manager = mp.Manager()
    res_queue = manager.Queue()
    seed_list = rng_main.integers(0, 2**32, size=n_boot)
    pool = mp.Pool(processes=n_workers)

    for s in seed_list:
        pool.apply_async(_boot_worker, args=(X, s, res_queue))
    pool.close()

    boot_nclss = []
    pbar = tqdm(total=n_boot, desc=f"Bootstrap sampling ({genus_name})", leave=True, ncols=100, unit='B', unit_scale=True)
    for _ in range(n_boot):
        nc = res_queue.get()
        boot_nclss.append(nc)
        pbar.update(1)
    pbar.close()
    pool.join()

    boot_nclss = np.array(boot_nclss)
    real_ncls_count = np.sum(boot_nclss == real_ncls)
    real_ncls_frac = real_ncls_count / n_boot

    stat_dict = {
        "real_threshold": real_t,
        "real_ncls": real_ncls,
        "real_sil": real_sil,
        "real_ncls_count": real_ncls_count,
        "real_ncls_frac": real_ncls_frac,
        "boot_ncls_raw": boot_nclss
    }
    return real_t, real_ncls, real_sil, real_labels, boot_nclss, real_ncls_count, real_ncls_frac, stat_dict

In [3]:
summary_rows = []
os.makedirs(BASE_FOLDER, exist_ok=True)

for genus_name in keep_genus:
    folder = os.path.join(BASE_FOLDER, genus_name)
    os.makedirs(folder, exist_ok=True)

    src_path = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}/replicon-plasmid_fraction-self_bitscore_statistics.csv'
    df = pd.read_csv(src_path)
    df['log10_size'] = np.log10(df['size'])
    
    raw_size_thresh = process_replicon_data(df)
    log10_size_mean = df['log10_size'].mean()
    log10_size_std = df['log10_size'].std()
    std_size_thresh = (np.log10(raw_size_thresh) - log10_size_mean) / log10_size_std

    raw_plasmidness_mean = df[plasmidness_col].mean()
    raw_plasmidness_std = df[plasmidness_col].std()
    raw_plasmidness_thresh = RAW_PLASMIDNESS_THRESHOLD
    std_plasmidness_thresh = (raw_plasmidness_thresh - raw_plasmidness_mean) / raw_plasmidness_std

    df['std_log10_size'] = (df['log10_size'] - log10_size_mean) / log10_size_std
    df['std_plasmidness'] = (df[plasmidness_col] - raw_plasmidness_mean) / raw_plasmidness_std

    X = df[["std_log10_size", "std_plasmidness"]].values
    n = X.shape[0]

    real_t, real_ncls, real_sil, labels, boot_nclss, real_ncls_count, real_ncls_frac, stat_dict = bootstrap_test(genus_name, X, N_BOOT, RANDOM_SEED)

    out_df = pd.DataFrame({
        "std_log10_size": X[:, 0],
        "std_plasmidness": X[:, 1],
        "cluster_label": labels
    })
    out_path = os.path.join(folder, f"{genus_name}_cluster_result.csv")
    out_df.to_csv(out_path, index=False)

    boot_df = pd.DataFrame({"boot_nclss": boot_nclss})
    boot_path = os.path.join(folder, f"{genus_name}_boot_ncls.csv")
    boot_df.to_csv(boot_path, index=False)

    summary_rows.append({
        "genus": genus_name,
        "sample_size": n,
        "raw_size_threshold": raw_size_thresh,
        "std_size_threshold": std_size_thresh,
        "raw_plasmidness_threshold": raw_plasmidness_thresh,
        "std_plasmidness_threshold": std_plasmidness_thresh,
        "real_threshold": stat_dict["real_threshold"],
        "real_ncls": stat_dict["real_ncls"],
        "real_sil": stat_dict["real_sil"],
        "real_ncls_count": stat_dict["real_ncls_count"],
        "repeat_ratio": stat_dict["real_ncls_frac"]
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_SUMMARY_CSV, index=False)

Bootstrap sampling (Escherichia): 100%|█████████████████████████| 1.00k/1.00k [21:31<00:00, 1.29s/B]
Bootstrap sampling (Klebsiella): 100%|██████████████████████████| 1.00k/1.00k [20:32<00:00, 1.23s/B]
Bootstrap sampling (Staphylococcus): 100%|██████████████████████| 1.00k/1.00k [01:48<00:00, 9.20B/s]
Bootstrap sampling (Pseudomonas): 100%|█████████████████████████| 1.00k/1.00k [00:50<00:00, 19.7B/s]
Bootstrap sampling (Bacillus): 100%|████████████████████████████| 1.00k/1.00k [01:07<00:00, 14.7B/s]
Bootstrap sampling (Salmonella): 100%|██████████████████████████| 1.00k/1.00k [01:18<00:00, 12.7B/s]
Bootstrap sampling (Streptococcus): 100%|███████████████████████| 1.00k/1.00k [00:18<00:00, 52.9B/s]
Bootstrap sampling (Streptomyces): 100%|████████████████████████| 1.00k/1.00k [00:26<00:00, 37.2B/s]
Bootstrap sampling (Acinetobacter): 100%|███████████████████████| 1.00k/1.00k [01:03<00:00, 15.8B/s]
Bootstrap sampling (Enterococcus): 100%|████████████████████████| 1.00k/1.00k [00:50<00:00,